<a href="https://colab.research.google.com/github/RoihansLab/Data-Mining-Project/blob/main/youtube_comment_sentiment/Classification/classification_modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Klasifikasi Sikap Publik terhadap AI yang Menggantikan Pekerjaan Manusia Berdasarkan Komentar YouTube**

**Mata Kuliah:** Data Mining

**Nama Anggota Kelompok:**

1.   Muhammad Rafa Enrico (Ketua - 4.33.24.2.15)
2.   Roihan Saputra (4.33.24.2.20)

**Kelas TI-3C**

**Program Studi:** D4 Teknologi Rekayasa Komputer, Politeknik Negeri Semarang

---
## Gambaran Umum Notebook
Notebook ini melanjutkan tahap *Data Preparation* (Bab 1-5) dan membangun model klasifikasi sikap publik dengan 3 label: **positif**, **negatif**, dan **perlu ditinjau**.

| Bab | Tahap | Isi |
|---|---|---|
| 6 | Text Preprocessing | case folding, removing, filtering (stopword), tokenizing, stemming |
| 7 | Ekstraksi Fitur | pembagian data 80:20, TF-IDF, embedding e5-large, penggabungan fitur |
| 8 | Klasifikasi | Logistic Regression dengan pengaturan awal |
| 9 | Optimasi & Evaluasi | Bayesian Search (Optuna), evaluasi metrik, cek overfitting/underfitting, dan robustness |
| 10 | Kesimpulan | hasil akhir, keterbatasan, dan rekomendasi |

**Model akhir:** e5-large + TF-IDF + Logistic Regression, di-*tuning* dengan Bayesian Search.

# **1. Import Library & Setup Environment**

Seluruh library untuk preprocessing teks, ekstraksi fitur, klasifikasi, dan optimasi disiapkan di sel ini. Semua sumber acak dikunci dengan angka **42** (`random_state=42`) agar hasil dapat direproduksi.

In [ ]:
# Pasang library jika belum terinstal
%pip install Sastrawi nltk sentence-transformers optuna scikit-learn scipy matplotlib torch --quiet

import os
import re
import random
import string

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Text preprocessing
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Ekstraksi fitur dan pembagian data
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from scipy.sparse import hstack, csr_matrix

# Klasifikasi, metrik, dan optimasi
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, f1_score, accuracy_score, precision_score, recall_score,
                             cohen_kappa_score, ConfusionMatrixDisplay)
import torch
import optuna

# Kunci random seed untuk reprodusibilitas
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

perangkat = "cuda" if torch.cuda.is_available() else "cpu"
print("Setup library berhasil.")
print("Perangkat komputasi:", torch.cuda.get_device_name(0) if perangkat == "cuda" else "CPU")

# **6. Text Preprocessing**

Bab 3 hanya membersihkan data mentah. Bab ini mengolah **teks komentar** supaya siap dibaca TF-IDF, dengan urutan:

**6.2 case folding → 6.3 removing → 6.4 filtering (stopword) → 6.5 tokenizing → 6.6 stemming**

Hasil olahan disimpan di kolom baru `teks` (dan `token`). Kolom `comment` tetap asli, karena dipakai oleh embedding e5-large di Bab 7.3.

## **6.1 Memuat Data Berlabel**

**Sel ini:** memuat data berlabel hasil Bab 5 (`youtube_comments_label_final.csv`), lalu menampilkan jumlah komentar dan sebaran labelnya. Lokasi berkas dicari otomatis, jadi sel yang sama bisa dijalankan di Google Colab maupun laptop.

In [ ]:
for path in ["/content/youtube_comments_label_final.csv", "../../datasets/youtube_comments_label_final.csv", "datasets/youtube_comments_label_final.csv"]:
    if os.path.exists(path):
        break

df = pd.read_csv(path)
print("Data dimuat dari:", path)
print("Jumlah komentar:", len(df), "| nilai kosong:", df.isna().sum().sum())
print(df["label"].value_counts())
df.head()

**Hasil:**

- Ada **3.935 komentar** dan 7 kolom, **tidak ada nilai kosong**.
- Kolom yang dipakai: `comment` (teks asli), `bahasa` (untuk stemming), dan `label` (jawaban). Kolom `sumber_label` hanya mencatat asal label (manual atau Claude).
- Sebaran label: perlu ditinjau 2.342 (59,5%), negatif 938 (23,8%), positif 655 (16,6%). Labelnya **tidak seimbang**, jadi model memakai `class_weight="balanced"` dan dinilai dengan **macro-F1**.

## **6.2 Case Folding**

Semua huruf diubah menjadi huruf kecil, supaya "AI", "Ai", dan "ai" dianggap kata yang sama.

**Sel ini:** mengubah kolom `comment` menjadi huruf kecil dan menyimpannya di kolom `teks`.

In [ ]:
df["teks"] = df["comment"].str.lower()

print("Komentar yang tadinya punya huruf besar:", (df["comment"] != df["teks"]).sum())
df[["comment", "teks"]].head()

**Hasil:** 3.414 dari 3.935 komentar punya huruf besar, dan sekarang semuanya sudah menjadi huruf kecil di kolom `teks`.

## **6.3 Removing**

Membuang bagian teks yang tidak dipakai TF-IDF, lalu merapikan spasi:

| Yang dibuang | Pola (regex) | Contoh |
|---|---|---|
| URL | `https?://\S+` atau `www.\S+` | `https://youtu.be/...` |
| semua karakter selain huruf a-z dan spasi | `[^a-z\s]` | angka `2030`, tanda baca `?!.,`, emoji 😭, simbol `#` |
| spasi berlebih | dipecah per kata lalu digabung dengan satu spasi | `"ai   bodoh"` → `"ai bodoh"` |

Karakter yang dibuang diganti spasi (bukan dihapus), supaya kata tidak menempel, misalnya `"ai,robot"` menjadi `"ai robot"`, bukan `"airobot"`.

**Sel ini:** membersihkan teks, lalu mengecek jumlah komentar yang masih mengandung karakter selain huruf dan spasi (harus 0).

In [ ]:
print("Sebelum: komentar berisi angka/tanda baca/emoji/simbol:", df["teks"].str.contains(r"[^a-z\s]").sum())

df["teks"] = df["teks"].str.replace(r"https?://\S+|www\.\S+", " ", regex=True)   # URL
df["teks"] = df["teks"].str.replace(r"[^a-z\s]", " ", regex=True)                # angka, tanda baca, emoji, simbol
df["teks"] = df["teks"].str.split().str.join(" ")                                 # spasi berlebih

print("Sesudah:", df["teks"].str.contains(r"[^a-z\s]").sum())
print("Komentar yang menjadi kosong:", (df["teks"] == "").sum())
df[["comment", "teks"]].head()

**Hasil:** sebelum dibersihkan, **3.456 komentar** mengandung angka, tanda baca, emoji, atau simbol. Sesudah dibersihkan, jumlahnya **0**, dan tidak ada komentar yang menjadi kosong.

## **6.4 Filtering (Stopword Removal)**

Membuang kata umum yang tidak membawa makna, misalnya "yang", "dan", "the", "is". Daftarnya gabungan stopword bahasa Indonesia (Sastrawi) dan bahasa Inggris (scikit-learn).

**Kata negasi tetap disimpan**, karena membalik makna kalimat: "AI **tidak** akan menggantikan pekerjaan" berbeda jauh dengan "AI akan menggantikan pekerjaan".

**Sel ini:** menyusun daftar stopword tanpa kata negasi, membuang stopword dari setiap komentar, lalu membuang komentar yang menjadi kosong.

In [ ]:
negasi = {"tidak", "belum", "nggak", "not", "no", "nor", "never", "nothing", "none", "cannot"}
stopword = (set(StopWordRemoverFactory().get_stop_words()) | set(ENGLISH_STOP_WORDS)) - negasi
print("Jumlah stopword:", len(stopword))

sebelum = df["teks"].str.split().str.len().mean()
df["teks"] = df["teks"].apply(lambda t: " ".join(kata for kata in t.split() if kata not in stopword))
print(f"Rata-rata jumlah kata per komentar: {sebelum:.1f} -> {df['teks'].str.split().str.len().mean():.1f}")

# Komentar yang isinya stopword semua menjadi kosong, jadi dibuang
print("Komentar kosong:", (df["teks"] == "").sum())
df = df[df["teks"] != ""].reset_index(drop=True)
print("Sisa data:", len(df))

**Hasil:**

- Ada 431 stopword. Rata-rata komentar menyusut dari 30,9 kata menjadi 18,0 kata (sekitar 42% kata terbuang).
- 8 komentar menjadi kosong karena isinya stopword semua (misalnya "Indeed, it was" dan "Been there, done that."), jadi dibuang. **Sisa data: 3.927 komentar.**

## **6.5 Tokenizing**

Memecah setiap komentar menjadi daftar kata (token). Contoh: `"ai tidak butuh makan"` → `["ai", "tidak", "butuh", "makan"]`.

**Sel ini:** memecah teks menjadi token di kolom `token`, lalu menghitung jumlah token dan kata unik.

In [ ]:
df["token"] = df["teks"].str.split()

semua_token = [kata for token in df["token"] for kata in token]
print("Jumlah token:", len(semua_token), "| kata unik:", len(set(semua_token)))
df[["teks", "token"]].head()

**Hasil:** 3.927 komentar dipecah menjadi **70.745 token** dengan **13.629 kata unik**.

## **6.6 Stemming**

Mengubah setiap kata ke bentuk dasarnya, supaya kata yang maknanya sama dihitung sebagai satu kata. Contoh: "pengusaha" → "usaha", "mengganti" → "ganti", "approaching" → "approach".

Stemmer dipilih sesuai bahasa komentar: **Sastrawi** untuk bahasa Indonesia dan **Porter** untuk bahasa Inggris. Proses ini sekitar 2-3 menit.

**Sel ini:** menjalankan stemming per kata sesuai bahasa komentar, lalu menyusun ulang kolom `teks` dari token hasil stemming.

In [ ]:
stem_id = StemmerFactory().create_stemmer()   # bahasa Indonesia
stem_en = PorterStemmer()                      # bahasa Inggris


def stem(kata, bahasa):
    return stem_id.stem(kata) if bahasa == "Indonesia" else stem_en.stem(kata)


df["token"] = [[stem(kata, bahasa) for kata in token] for token, bahasa in zip(df["token"], df["bahasa"])]
df["teks"] = df["token"].str.join(" ")         # teks akhir untuk TF-IDF

print("Kata unik setelah stemming:", len({kata for token in df["token"] for kata in token}))
df[["comment", "teks"]].head()

**Hasil:** jumlah kata unik turun dari 13.629 menjadi **9.848** (sekitar 28% lebih sedikit), karena kata-kata berimbuhan digabung ke kata dasarnya. Contoh: "semua pengusaha berpikiran sama mengganti manusia" → "semua usaha pikir sama ganti manusia".

Catatan: stemmer bahasa Inggris (Porter) kadang menghasilkan potongan yang bukan kata utuh, misalnya "replacing" → "replac". Ini wajar, karena yang penting kata-kata sejenis menjadi sama.

# **7. Ekstraksi Fitur**

Model tidak bisa membaca kata, jadi setiap komentar diubah menjadi deretan angka (fitur). Model akhir memakai **dua jenis fitur yang digabung**:

| Fitur | Masukan | Yang ditangkap |
|---|---|---|
| **TF-IDF** (unigram + bigram) | kolom `teks` (hasil Bab 6) | kata kunci penting, misalnya "takut", "phk", "not replac" |
| **e5-large** (embedding *pre-trained*) | kolom `comment` (teks asli) | makna seluruh kalimat, termasuk konteks dan nada |

Urutannya: 7.1 pembagian data → 7.2 TF-IDF → 7.3 embedding e5-large → 7.4 penggabungan fitur.

## **7.1 Pembagian Data Latih dan Uji**

Data **dibagi dulu** sebelum fitur dibuat: 80% data latih dan 20% data uji, dengan proporsi ketiga label yang sama (`stratify`). TF-IDF dan model hanya belajar dari data latih, supaya data uji benar-benar belum pernah dilihat.

**Sel ini:** membagi data, lalu menyimpan nomor baris data latih (`latih`) dan data uji (`uji`).

In [ ]:
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["label"])

latih = df_train.index.to_numpy()   # nomor baris data latih
uji = df_test.index.to_numpy()      # nomor baris data uji
teks = df["teks"].to_numpy()        # teks hasil Bab 6
y = df["label"].to_numpy()          # label

print("Data latih:", len(latih), "| data uji:", len(uji))
pd.DataFrame({"latih": df_train["label"].value_counts(), "uji": df_test["label"].value_counts()})

**Hasil:** data latih **3.141 komentar** dan data uji **786 komentar**, dengan proporsi label yang sama:

| Label | Latih | Uji |
|---|---|---|
| perlu ditinjau | 1.867 | 467 |
| negatif | 750 | 188 |
| positif | 524 | 131 |

## **7.2 TF-IDF**

**TF-IDF** memberi nilai tinggi pada kata yang sering muncul di sebuah komentar tetapi jarang muncul di komentar lain, sehingga kata yang khas untuk suatu sikap mendapat bobot besar. Yang dihitung bukan hanya satu kata (**unigram**) tetapi juga pasangan dua kata berurutan (**bigram**), misalnya "not replac", karena maknanya bisa berbeda dari kata-katanya sendiri.

Pengaturan awal (nanti di-*tuning* di Bab 9):
- `ngram_range=(1, 2)`: unigram + bigram.
- `min_df=2`: kata harus muncul minimal di 2 komentar.
- `sublinear_tf=True`: kata yang muncul berkali-kali dalam satu komentar diredam.

**Sel ini:** melatih TF-IDF **hanya pada data latih**.

In [ ]:
tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
tfidf.fit(teks[latih])

print("Jumlah kata dan pasangan kata di kosakata TF-IDF:", len(tfidf.vocabulary_))

## **7.3 Embedding e5-large**

TF-IDF hanya menghitung kata, jadi "takut" dan "khawatir" dianggap tidak berhubungan. **multilingual-e5-large** adalah model embedding yang sudah dilatih dari jutaan kalimat dalam banyak bahasa (termasuk Indonesia dan Inggris). Setiap komentar diubah menjadi **1.024 angka** yang mewakili makna seluruh kalimat, sehingga komentar yang maknanya mirip punya angka yang mirip walaupun kata-katanya berbeda.

- Masukannya kolom `comment` (teks asli), karena e5 memanfaatkan tanda baca, huruf besar, dan kata sambung.
- e5 mewajibkan awalan `"query: "` pada setiap teks.
- e5 **tidak dilatih ulang** (hanya dipakai), jadi semua komentar boleh diubah sekaligus tanpa membocorkan informasi data uji. Prosesnya sekitar 2-3 menit di GPU (lebih lama di CPU).

**Sel ini:** memuat e5-large dan mengubah setiap komentar menjadi 1.024 angka.

In [ ]:
e5 = SentenceTransformer("intfloat/multilingual-e5-large", device=perangkat)
X_e5 = e5.encode(["query: " + t for t in df["comment"]], batch_size=16, normalize_embeddings=True, show_progress_bar=True)
print("Ukuran fitur e5-large:", X_e5.shape)

## **7.4 Penggabungan Fitur**

Kedua fitur ditempel berdampingan (`hstack`), sehingga setiap komentar diwakili angka e5-large **dan** angka TF-IDF. TF-IDF dikalikan **bobot** (awalnya 0,5) supaya pengaruhnya seimbang dengan e5-large. Bobot ini juga di-*tuning* di Bab 9.

**Sel ini:** membuat fungsi `gabung_fitur`, lalu membuat fitur data latih dan data uji.

In [ ]:
def gabung_fitur(tfidf, idx, bobot_tfidf):
    """Tempel fitur e5-large dan TF-IDF untuk baris-baris idx."""
    return hstack([csr_matrix(X_e5[idx]), tfidf.transform(teks[idx]) * bobot_tfidf]).tocsr()


X_latih = gabung_fitur(tfidf, latih, 0.5)
X_uji = gabung_fitur(tfidf, uji, 0.5)
print("Ukuran fitur latih:", X_latih.shape, "| uji:", X_uji.shape)

**Hasil:**

| Fitur | Jumlah angka per komentar |
|---|---|
| TF-IDF (unigram + bigram, `min_df=2`) | 7.903 |
| e5-large | 1.024 |
| **Gabungan** | **8.927** |

# **8. Klasifikasi (Logistic Regression)**

**Logistic Regression** menghitung peluang setiap komentar masuk ke masing-masing label dari fitur gabungan, lalu memilih label dengan peluang terbesar. Model ini cocok untuk fitur berdimensi besar seperti e5-large + TF-IDF, cepat dilatih, dan hasilnya mudah dijelaskan.

Pengaturan awal:
- `C=3`: seberapa ketat model mengikuti data latih (kecil = lebih sederhana, besar = lebih detail).
- `class_weight="balanced"`: label yang sedikit (positif dan negatif) diberi bobot lebih besar, supaya model tidak hanya menebak "perlu ditinjau".

Hasil bab ini menjadi **pembanding** untuk hasil optimasi di Bab 9.

**Sel ini:** melatih Logistic Regression dengan pengaturan awal, lalu menilainya di data uji.

In [ ]:
lr = LogisticRegression(C=3, class_weight="balanced", max_iter=3000, random_state=42)
lr.fit(X_latih, y[latih])
pred_awal = lr.predict(X_uji)

print("Macro-F1:", round(f1_score(y[uji], pred_awal, average="macro"), 3), "| Akurasi:", round(accuracy_score(y[uji], pred_awal), 3))
print(classification_report(y[uji], pred_awal, digits=3))

**Hasil (pengaturan awal):** macro-F1 **0,651** dan akurasi **0,711**.

| Label | Precision | Recall | F1 |
|---|---|---|---|
| negatif | 0,566 | 0,638 | 0,600 |
| perlu ditinjau | 0,868 | 0,775 | 0,819 |
| positif | 0,490 | 0,588 | 0,535 |

Label "perlu ditinjau" paling mudah ditebak karena contohnya paling banyak. Positif paling sulit karena contohnya paling sedikit (524 komentar latih).

Catatan: angka di bawah berasal dari percobaan dengan embedding e5-large yang disimpan dalam float16, sehingga hasil sel di atas bisa berbeda sedikit (sekitar 0,01).

# **9. Optimasi dan Evaluasi**

Pengaturan di Bab 7-8 masih tebakan awal. Bab ini mencari pengaturan terbaik dengan **Bayesian Search** (library **Optuna**). Bedanya dengan mencoba semua kombinasi: setiap percobaan berikutnya dipilih berdasarkan hasil percobaan sebelumnya, sehingga lebih cepat menemukan pengaturan yang bagus.

| Pengaturan | Artinya | Rentang yang dicari |
|---|---|---|
| `C` | seberapa ketat model mengikuti data latih | 0,01-100 |
| `class_weight` | label kecil diberi bobot lebih atau tidak | balanced / tidak |
| `ngram` | TF-IDF per kata saja (1), atau kata + pasangan kata (2) | 1 / 2 |
| `min_df` | kata minimal muncul di berapa komentar | 1 / 2 / 3 |
| `sublinear` | kata yang muncul berkali-kali diredam atau tidak | ya / tidak |
| `bobot_tfidf` | seberapa besar pengaruh TF-IDF dibanding e5-large | 0,1-2 |

**Aturan supaya skornya jujur:**
1. Setiap percobaan dinilai dengan **5-fold cross-validation di data latih saja**: data latih dibagi 5, lalu 5 kali model belajar dari 4 bagian dan diuji di 1 bagian sisanya.
2. **TF-IDF dilatih ulang di setiap fold**, hanya dari 4 bagian latihnya, supaya kosakata bagian validasi tidak bocor.
3. Data uji (20%) **tidak ikut memilih pengaturan**. Data uji baru dipakai **sekali di akhir**.

Isi bab ini: **9.1** Bayesian Search → **9.2** evaluasi metrik model akhir → **9.3** cek overfitting dan underfitting → **9.4** cek robustness.

## **9.1 Bayesian Search**

**Sel ini:** membuat dua fungsi.
- `latih_dan_tebak`: melatih TF-IDF + Logistic Regression dengan pengaturan `p` di baris `idx_latih`, lalu menebak label baris `idx_tebak`.
- `skor_cv`: menghitung rata-rata macro-F1 5-fold cross-validation **di data latih saja**.

In [ ]:
def latih_dan_tebak(p, idx_latih, idx_tebak):
    tfidf = TfidfVectorizer(ngram_range=(1, p["ngram"]), min_df=p["min_df"], sublinear_tf=p["sublinear"])
    tfidf.fit(teks[idx_latih])                               # TF-IDF hanya belajar dari idx_latih
    lr = LogisticRegression(C=p["C"], class_weight=p["class_weight"], max_iter=3000, random_state=42)
    lr.fit(gabung_fitur(tfidf, idx_latih, p["bobot_tfidf"]), y[idx_latih])
    return lr.predict(gabung_fitur(tfidf, idx_tebak, p["bobot_tfidf"]))


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)


def skor_cv(p):
    skor = []
    for a, b in cv.split(latih, y[latih]):                 # a = 4 bagian latih, b = 1 bagian validasi
        pred = latih_dan_tebak(p, latih[a], latih[b])
        skor.append(f1_score(y[latih[b]], pred, average="macro"))
    return np.mean(skor)

**Sel ini:** menjalankan Bayesian Search sebanyak 100 percobaan dan menampilkan pengaturan terbaiknya. Sel ini sekitar 10-15 menit.

In [ ]:
def tujuan(trial):
    p = {"C": trial.suggest_float("C", 0.01, 100, log=True),
         "class_weight": trial.suggest_categorical("class_weight", ["balanced", None]),
         "ngram": trial.suggest_categorical("ngram", [1, 2]),
         "min_df": trial.suggest_int("min_df", 1, 3),
         "sublinear": trial.suggest_categorical("sublinear", [True, False]),
         "bobot_tfidf": trial.suggest_float("bobot_tfidf", 0.1, 2.0)}
    return skor_cv(p)


optuna.logging.set_verbosity(optuna.logging.WARNING)
studi = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
studi.optimize(tujuan, n_trials=100, show_progress_bar=True)

p_terbaik = studi.best_params
print("Pengaturan terbaik:", p_terbaik)
print("Macro-F1 CV:", round(studi.best_value, 3))

**Sel ini:** melatih ulang model dengan pengaturan terbaik di seluruh data latih, lalu menilainya **sekali** di data uji dan membandingkannya dengan pengaturan awal (Bab 8).

In [ ]:
p_awal = {"C": 3, "class_weight": "balanced", "ngram": 2, "min_df": 2, "sublinear": True, "bobot_tfidf": 0.5}
pred_akhir = latih_dan_tebak(p_terbaik, latih, uji)

pd.DataFrame([
    {"Pengaturan": "Awal (Bab 8)", "Macro-F1 CV": skor_cv(p_awal),
     "Macro-F1 uji": f1_score(y[uji], pred_awal, average="macro"), "Akurasi uji": accuracy_score(y[uji], pred_awal)},
    {"Pengaturan": "Bayesian Search", "Macro-F1 CV": studi.best_value,
     "Macro-F1 uji": f1_score(y[uji], pred_akhir, average="macro"), "Akurasi uji": accuracy_score(y[uji], pred_akhir)},
]).round(3)

## **9.2 Evaluasi Metrik Model Akhir**

| Metrik | Artinya |
|---|---|
| **Akurasi** | persentase komentar yang ditebak benar |
| **Precision** | dari komentar yang ditebak label X, berapa persen yang benar-benar X |
| **Recall** | dari komentar yang sebenarnya X, berapa persen yang berhasil ditebak X |
| **F1** | rata-rata harmonis precision dan recall |
| **Macro-F1** | rata-rata F1 ketiga label dengan bobot sama (**metrik utama**, karena labelnya tidak seimbang) |
| **Weighted-F1** | rata-rata F1 yang dibobot jumlah komentar tiap label |
| **Cohen's kappa** | kecocokan tebakan dengan label setelah dikurangi kecocokan karena kebetulan (0 = asal tebak, 1 = sempurna) |

**Sel ini:** menghitung semua metrik model akhir di data uji.

In [ ]:
pd.Series({
    "Akurasi": accuracy_score(y[uji], pred_akhir),
    "Macro precision": precision_score(y[uji], pred_akhir, average="macro"),
    "Macro recall": recall_score(y[uji], pred_akhir, average="macro"),
    "Macro-F1": f1_score(y[uji], pred_akhir, average="macro"),
    "Weighted-F1": f1_score(y[uji], pred_akhir, average="weighted"),
    "Cohen's kappa": cohen_kappa_score(y[uji], pred_akhir),
}).round(3)

**Sel ini:** menampilkan skor per label dan confusion matrix model akhir.

In [ ]:
print(classification_report(y[uji], pred_akhir, digits=3))

ConfusionMatrixDisplay.from_predictions(y[uji], pred_akhir, cmap="Blues", xticks_rotation=20)
plt.title("Confusion Matrix Model Akhir")
plt.show()

**Hasil:**

**Pengaturan terbaik Bayesian Search** (dari 100 percobaan):

| Pengaturan | Awal (Bab 8) | **Bayesian Search** |
|---|---|---|
| `C` | 3 | **4,00** |
| `class_weight` | balanced | **balanced** |
| `ngram` | 2 | **2** |
| `min_df` | 2 | **1** |
| `sublinear` | ya | **ya** |
| `bobot_tfidf` | 0,5 | **0,62** |

| Pengaturan | Macro-F1 CV (data latih) | Macro-F1 uji | Akurasi uji |
|---|---|---|---|
| Awal (Bab 8) | 0,665 | 0,651 | 0,711 |
| **Bayesian Search** | **0,668** | **0,660** | **0,723** |

**Metrik model akhir di data uji:**

| Metrik | Nilai |
|---|---|
| Akurasi | 0,723 |
| Macro precision | 0,652 |
| Macro recall | 0,671 |
| **Macro-F1** | **0,660** |
| Weighted-F1 | 0,728 |
| Cohen's kappa | 0,522 |

| Label | Precision | Recall | F1 |
|---|---|---|---|
| negatif | 0,573 | 0,644 | 0,607 |
| perlu ditinjau | 0,865 | 0,797 | 0,829 |
| positif | 0,517 | 0,573 | 0,543 |

**Confusion matrix** (baris = label asli, kolom = tebakan model):

| Asli \ Tebakan | negatif | perlu ditinjau | positif |
|---|---|---|---|
| **negatif** | **121** | 33 | 34 |
| **perlu ditinjau** | 59 | **372** | 36 |
| **positif** | 31 | 25 | **75** |

- Bayesian Search menaikkan macro-F1 uji dari 0,651 menjadi **0,660** dan akurasi dari 0,711 menjadi **0,723**. Kenaikannya kecil, karena pengaturan awal sudah cukup dekat dengan yang terbaik.
- Pengaturan yang paling berpengaruh adalah `class_weight="balanced"` dan `min_df=1` (semua kata dipakai).
- **Kappa 0,522** berarti kesepakatan model dengan label **sedang**. Angka ini setara dengan kecocokan label Claude dengan manusia di Bab 5.3 (kappa 0,506), jadi model sudah mendekati batas kualitas labelnya.
- Kesalahan terbanyak: 59 komentar "perlu ditinjau" ditebak negatif, serta negatif dan positif yang saling tertukar (34 dan 31 komentar), biasanya pada komentar sindiran atau yang nadanya campur.

Catatan: angka di bawah berasal dari percobaan dengan embedding e5-large yang disimpan dalam float16, sehingga hasil sel di atas bisa berbeda sedikit (sekitar 0,01).

## **9.3 Cek Overfitting dan Underfitting**

| Kondisi | Tanda | Artinya |
|---|---|---|
| **Underfitting** | skor data latih **dan** data uji sama-sama rendah | model terlalu sederhana, belum menangkap pola |
| **Overfitting** | skor data latih jauh lebih tinggi daripada data uji, dan skor uji turun kalau model dibuat lebih rumit | model menghafal data latih, termasuk *noise*-nya |
| **Good fit** | skor validasi/uji paling tinggi, dan tidak naik lagi kalau model dibuat lebih rumit atau lebih sederhana | keseimbangan yang paling baik |

Tiga cara cek yang dipakai:
1. **Skor data latih vs data uji** dengan pengaturan terbaik.
2. **Kurva validasi `C`**: `C` diubah dari kecil (model sederhana) sampai besar (model rumit). Skor validasi diambil dari 5-fold CV di data latih, jadi data uji tidak dipakai.
3. **Kurva belajar (*learning curve*)**: model dilatih dengan 20%, 40%, ..., 100% data latih, untuk melihat apakah menambah data masih menaikkan skor.

**Sel ini:** membandingkan skor data latih dan data uji, keseluruhan dan per label.

In [ ]:
pred_latih = latih_dan_tebak(p_terbaik, latih, latih)   # model menebak data latihnya sendiri
label = ["negatif", "perlu ditinjau", "positif"]

tabel_fit = pd.DataFrame({
    "Data latih": [accuracy_score(y[latih], pred_latih), f1_score(y[latih], pred_latih, average="macro"),
                   *f1_score(y[latih], pred_latih, average=None, labels=label)],
    "Data uji": [accuracy_score(y[uji], pred_akhir), f1_score(y[uji], pred_akhir, average="macro"),
                 *f1_score(y[uji], pred_akhir, average=None, labels=label)],
}, index=["Akurasi", "Macro-F1"] + ["F1 " + l for l in label])
tabel_fit["Selisih"] = tabel_fit["Data latih"] - tabel_fit["Data uji"]
tabel_fit.round(3)

**Sel ini:** membuat kurva validasi `C`. Pengaturan lain tetap memakai hasil Bayesian Search. Sel ini sekitar 2-3 menit.

In [ ]:
daftar_C = [0.01, 0.1, 1, round(p_terbaik["C"], 2), 30, 100]
kurva_C = []
for C in daftar_C:
    p = {**p_terbaik, "C": C}
    kurva_C.append({"C": C,
                    "Macro-F1 latih": f1_score(y[latih], latih_dan_tebak(p, latih, latih), average="macro"),
                    "Macro-F1 validasi (CV)": skor_cv(p)})
kurva_C = pd.DataFrame(kurva_C)

plt.plot(kurva_C["C"], kurva_C["Macro-F1 latih"], "o-", label="Data latih")
plt.plot(kurva_C["C"], kurva_C["Macro-F1 validasi (CV)"], "o-", label="Validasi (5-fold CV)")
plt.axvline(p_terbaik["C"], color="gray", linestyle="--", label="C terbaik")
plt.xscale("log")
plt.xlabel("C (kiri = lebih sederhana, kanan = lebih rumit)")
plt.ylabel("Macro-F1")
plt.title("Kurva Validasi C")
plt.legend()
plt.show()
kurva_C.round(3)

**Sel ini:** membuat kurva belajar. Sebagian data latih diambil secara *stratified* (`random_state=42`), model dilatih dengan pengaturan terbaik, lalu dinilai di data latihnya sendiri dan di data uji. Data uji di sini **hanya untuk melihat**, tidak dipakai memilih pengaturan.

In [ ]:
kurva_belajar = []
for porsi in [0.2, 0.4, 0.6, 0.8, 1.0]:
    if porsi < 1:
        sebagian, _ = train_test_split(latih, train_size=porsi, random_state=42, stratify=y[latih])
    else:
        sebagian = latih
    kurva_belajar.append({"Porsi data latih": porsi, "Jumlah komentar": len(sebagian),
                          "Macro-F1 latih": f1_score(y[sebagian], latih_dan_tebak(p_terbaik, sebagian, sebagian), average="macro"),
                          "Macro-F1 uji": f1_score(y[uji], latih_dan_tebak(p_terbaik, sebagian, uji), average="macro")})
kurva_belajar = pd.DataFrame(kurva_belajar)

plt.plot(kurva_belajar["Jumlah komentar"], kurva_belajar["Macro-F1 latih"], "o-", label="Data latih")
plt.plot(kurva_belajar["Jumlah komentar"], kurva_belajar["Macro-F1 uji"], "o-", label="Data uji")
plt.xlabel("Jumlah komentar latih")
plt.ylabel("Macro-F1")
plt.title("Kurva Belajar")
plt.legend()
plt.show()
kurva_belajar.round(3)

**Hasil:**

**1. Data latih vs data uji**

| | Data latih | Data uji | Selisih |
|---|---|---|---|
| Akurasi | 0,950 | 0,723 | 0,228 |
| Macro-F1 | 0,947 | 0,660 | 0,287 |
| F1 negatif | 0,935 | 0,607 | 0,328 |
| F1 perlu ditinjau | 0,957 | 0,829 | 0,128 |
| F1 positif | 0,950 | 0,543 | 0,406 |

**2. Kurva validasi `C`**

| C | Macro-F1 latih | Macro-F1 validasi (CV) | Kondisi |
|---|---|---|---|
| 0,01 | 0,656 | 0,598 | underfitting (dua-duanya rendah) |
| 0,1 | 0,696 | 0,618 | underfitting |
| 1 | 0,841 | 0,654 | mendekati terbaik |
| **4,00 (terpilih)** | **0,948** | **0,668** | **titik terbaik** |
| 30 | 0,999 | 0,656 | overfitting (latih hampir sempurna, validasi turun) |
| 100 | 1,000 | 0,651 | overfitting |

**3. Kurva belajar**

| Jumlah komentar latih | Macro-F1 latih | Macro-F1 uji |
|---|---|---|
| 628 (20%) | 0,989 | 0,611 |
| 1.256 (40%) | 0,974 | 0,642 |
| 1.884 (60%) | 0,961 | 0,656 |
| 2.512 (80%) | 0,952 | 0,654 |
| 3.141 (100%) | 0,947 | 0,660 |

**Kesimpulan cek fit:**
- **Tidak underfitting.** Skor latih tinggi (0,947), jadi model mampu menangkap pola data.
- **Pengaturan terpilih adalah titik good fit.** Pada kurva validasi, `C = 4,00` memberi skor validasi tertinggi (0,668). Model yang lebih sederhana (`C` kecil) skornya turun karena underfitting, dan model yang lebih rumit (`C` besar) juga turun karena overfitting.
- **Masih ada selisih latih-uji yang besar (sekitar 0,29)**, yaitu tanda *variance* atau overfitting ringan. Sumber utamanya adalah TF-IDF: setelah *tuning* (`min_df=1`, unigram + bigram) fiturnya menjadi 51.931 kata dan pasangan kata (total 52.955 fitur dengan e5-large), padahal komentar latih hanya 3.141, sehingga banyak kata yang hanya muncul di satu-dua komentar ikut dihafal model, ditambah label yang belum sepenuhnya konsisten (kappa Claude vs manusia 0,506). Memperkuat regularisasi (memperkecil `C`) tidak menolong, karena skor validasi malah turun.
- **Menambah data masih membantu.** Pada kurva belajar, skor uji naik dari 0,611 menjadi 0,660 dan selisihnya menyempit seiring bertambahnya data. Jadi cara paling efektif untuk mengurangi overfitting adalah **menambah data berlabel**, terutama label manual.

Catatan: angka di bawah berasal dari percobaan dengan embedding e5-large yang disimpan dalam float16, sehingga hasil sel di atas bisa berbeda sedikit (sekitar 0,01).

## **9.4 Cek Robustness**

Model disebut **robust** (tangguh) jika skornya **stabil**, tidak naik-turun jauh hanya karena data yang dipakai sedikit berbeda. Dua cara cek:
1. **Skor per fold**: 5-fold CV di data latih. Jika simpangan bakunya (*std*) kecil, model stabil di berbagai potongan data.
2. **Bootstrap data uji**: 786 komentar uji diambil ulang secara acak dengan pengembalian sebanyak 1.000 kali (`random_state=42`), lalu macro-F1 dihitung setiap kali. Rentang 95% hasilnya menunjukkan seberapa jauh skor uji bisa bergeser karena kebetulan.

**Sel ini:** menghitung skor per fold dan rentang bootstrap 95%.

In [ ]:
skor_fold = [f1_score(y[latih[b]], latih_dan_tebak(p_terbaik, latih[a], latih[b]), average="macro")
             for a, b in cv.split(latih, y[latih])]
print("Macro-F1 per fold:", np.round(skor_fold, 3))
print(f"Rata-rata: {np.mean(skor_fold):.3f} | simpangan baku: {np.std(skor_fold):.3f}")

acak = np.random.default_rng(42)
skor_boot = []
for _ in range(1000):
    i = acak.integers(0, len(uji), len(uji))             # ambil ulang data uji dengan pengembalian
    skor_boot.append(f1_score(y[uji][i], pred_akhir[i], average="macro"))
bawah, atas = np.percentile(skor_boot, [2.5, 97.5])
print(f"Macro-F1 uji: {f1_score(y[uji], pred_akhir, average='macro'):.3f} | rentang 95% bootstrap: {bawah:.3f} - {atas:.3f}")

**Hasil:**

| Cek | Hasil |
|---|---|
| Macro-F1 per fold (5-fold CV) | 0,683 / 0,655 / 0,661 / 0,653 / 0,690 |
| Rata-rata ± simpangan baku | **0,668 ± 0,015** |
| Macro-F1 data uji | 0,660 |
| Rentang 95% bootstrap data uji | **0,619 - 0,694** |

- **Model robust.** Skor di kelima fold berdekatan (0,653-0,690) dengan simpangan baku hanya 0,015, jadi skor tidak bergantung pada potongan data tertentu.
- Skor uji (0,660) **konsisten** dengan skor CV (0,668) dan berada di dalam rentang bootstrap. Artinya pengaturan tidak "kebetulan cocok" dengan data latih saja.
- Rentang bootstrap selebar sekitar 0,08 menunjukkan bahwa selisih skor kecil (misalnya 0,01) antar-pengaturan belum tentu berarti, karena masih di dalam variasi acak data uji.

Catatan: angka di bawah berasal dari percobaan dengan embedding e5-large yang disimpan dalam float16, sehingga hasil sel di atas bisa berbeda sedikit (sekitar 0,01).

# **10. Kesimpulan dan Rekomendasi**

## **10.1 Kesimpulan**

1. **Sebaran sikap publik (Bab 5.4).** Dari 3.935 komentar, 59,5% perlu ditinjau (tidak bersikap jelas atau *off-topic*). Di antara 1.593 komentar yang bersikap jelas, **negatif lebih banyak (58,9%)** daripada positif (41,1%): publik yang bersikap lebih banyak yang **khawatir atau menolak** AI yang menggantikan pekerjaan.
2. **Model terbaik.** Alur akhir adalah text preprocessing (case folding, removing, stopword dengan negasi tetap disimpan, tokenizing, stemming) → fitur **e5-large + TF-IDF** → **Logistic Regression** (`class_weight="balanced"`) → **Bayesian Search**. Hasilnya pada 20% data uji:

| Metrik | Nilai |
|---|---|
| Macro-F1 | **0,660** |
| Akurasi | **0,723** |
| Cohen's kappa | 0,522 |

3. **Good fit dan robust.** Pengaturan terpilih berada di titik terbaik kurva validasi (tidak underfitting, dan model yang lebih rumit justru overfitting). Skor stabil antar-fold (0,668 ± 0,015) dan konsisten dengan data uji. Masih ada selisih latih-uji sekitar 0,29, yang terutama disebabkan jumlah data dan konsistensi label.

## **10.2 Keterbatasan**

- Sebagian besar label (sekitar 89%) dibuat oleh Claude Opus 5.5, dengan kecocokan sedang terhadap manusia (kappa 0,506). Kualitas label ini menjadi batas atas skor model.
- Data latih relatif kecil (3.141 komentar) dibanding jumlah fitur model akhir (52.955, terutama dari TF-IDF), sehingga masih ada selisih skor latih-uji.
- Label positif paling sulit ditebak (F1 0,543) karena contohnya paling sedikit.

## **10.3 Rekomendasi**

1. **Tambah label manual.** Kurva belajar menunjukkan skor masih naik jika data bertambah. Cara paling efisien adalah *active learning*: melabeli manual komentar yang membuat model ragu atau yang paling mungkin negatif/positif, bukan komentar acak.
2. **Ukur kesepakatan antarmanusia.** Sebagian komentar dilabeli dua orang secara terpisah untuk mengetahui batas kesepakatan manusia sendiri, lalu panduan pelabelan dipertegas pada kasus yang sering berbeda.
3. **Fine-tuning model Transformer dengan GPU yang lebih besar** (misalnya IndoBERT, XLM-RoBERTa-large) setelah label bertambah dan lebih konsisten.